In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder.appName("ExpenseMonitoringSystem").getOrCreate()

In [0]:
data=[
(1,"Priyanka Sharma","priyankasharma@gmail.com","9876543210","Female",65500),
(2,"Ankit Sharma","ankitsharma@gmail.com","9876543211","Male",70000),
(3,"Sonakshi Sharma","sonakshisharma@gmail.com","9876543212","Female",60000),
(4,"Karan Sharma","karansharma@gmail.com","9876293837","Male",30000)
]
columns=["user_id","user_name","email","phone","gender","monthly_income"]
users_df=spark.createDataFrame(data,columns)

data=[
(1,1,"Priyanka Sharma","Food",450.0,"UPI","2026-03-25","Swiggy","2026-03"),
(2,1,"Priyanka Sharma","Travel",1200.0,"Card","2026-03-29","RedBus","2026-03"),
(3,2,"Ankit Sharma","Shopping",5000.0,"Card","2026-04-02","Amazon","2026-04"),
(4,3,"Sonakshi Sharma","Bills",2500.0,"UPI","2026-04-15","TNEB","2026-04"),
(5,2,"Ankit Sharma","Shopping",3000.0,"Credit Card","2026-04-23","Myntra","2026-04"),
(7,3,"Sonakshi Sharma","Bills",12000.0,"Cash","2026-05-05","House rent","2026-05"),
(8,2,"Ankit Sharma","Entertainment",199.0,"UPI","2026-05-07","Netflix","2026-05"),
(9,4,"Karan Sharma","Travel",5000.0,"Debit Card","2026-05-10","Hotel","2026-05")
]
columns=["expense_id","user_id","user_name","category","amount","payment_method","expense_date","merchant","month"]
expenses_df=spark.createDataFrame(data,columns)
expenses_df=expenses_df.withColumn("month",date_format(col("expense_date"),"yyyy-MM"))

In [0]:
display(users_df)
display(expenses_df)

user_id,user_name,email,phone,gender,monthly_income
1,Priyanka Sharma,priyankasharma@gmail.com,9876543210,Female,65500
2,Ankit Sharma,ankitsharma@gmail.com,9876543211,Male,70000
3,Sonakshi Sharma,sonakshisharma@gmail.com,9876543212,Female,60000
4,Karan Sharma,karansharma@gmail.com,9876293837,Male,30000


expense_id,user_id,user_name,category,amount,payment_method,expense_date,merchant,month
1,1,Priyanka Sharma,Food,450.0,UPI,2026-03-25,Swiggy,2026-03
2,1,Priyanka Sharma,Travel,1200.0,Card,2026-03-29,RedBus,2026-03
3,2,Ankit Sharma,Shopping,5000.0,Card,2026-04-02,Amazon,2026-04
4,3,Sonakshi Sharma,Bills,2500.0,UPI,2026-04-15,TNEB,2026-04
5,2,Ankit Sharma,Shopping,3000.0,Credit Card,2026-04-23,Myntra,2026-04
7,3,Sonakshi Sharma,Bills,12000.0,Cash,2026-05-05,House rent,2026-05
8,2,Ankit Sharma,Entertainment,199.0,UPI,2026-05-07,Netflix,2026-05
9,4,Karan Sharma,Travel,5000.0,Debit Card,2026-05-10,Hotel,2026-05


In [0]:
# Monthly user spending
monthly_spend=expenses_df.groupBy("user_id","user_name","month").agg(sum("amount").alias("monthly_spend"))
print("Monthly user spending")
display(monthly_spend)

Monthly user spending


user_id,user_name,month,monthly_spend
1,Priyanka Sharma,2026-03,1650.0
2,Ankit Sharma,2026-04,8000.0
3,Sonakshi Sharma,2026-04,2500.0
3,Sonakshi Sharma,2026-05,12000.0
2,Ankit Sharma,2026-05,199.0
4,Karan Sharma,2026-05,5000.0


In [0]:
# Join datasets
final_df=monthly_spend.join(users_df,on="user_id",how="inner")
final_df=final_df.drop(users_df.user_name)

# Calculate savings
final_df=final_df.withColumn("savings",col("monthly_income")-col("monthly_spend"))

# Generate alerts
final_df=final_df.withColumn("alert",when(col("savings")<10000,"High Spending").otherwise("Normal"))
print("Final ETL Report")
display(final_df)

Final ETL Report


user_id,user_name,month,monthly_spend,email,phone,gender,monthly_income,savings,alert
1,Priyanka Sharma,2026-03,1650.0,priyankasharma@gmail.com,9876543210,Female,65500,63850.0,Normal
2,Ankit Sharma,2026-04,8000.0,ankitsharma@gmail.com,9876543211,Male,70000,62000.0,Normal
3,Sonakshi Sharma,2026-04,2500.0,sonakshisharma@gmail.com,9876543212,Female,60000,57500.0,Normal
3,Sonakshi Sharma,2026-05,12000.0,sonakshisharma@gmail.com,9876543212,Female,60000,48000.0,Normal
2,Ankit Sharma,2026-05,199.0,ankitsharma@gmail.com,9876543211,Male,70000,69801.0,Normal
4,Karan Sharma,2026-05,5000.0,karansharma@gmail.com,9876293837,Male,30000,25000.0,Normal


In [0]:
# Save as Delta Table
final_df.write.format("delta").mode("overwrite").saveAsTable("final_report")

# Save as CSV file
final_df.toPandas().to_csv("final_report.csv")
print("Final report saved successfully")

Final report saved successfully
